# 🔩 GC10-DET — Modelo Unificado: Classificação + Detecção com YOLOv8

## O que este notebook resolve

O pipeline anterior usava **dois modelos separados e desconectados**:
- `EfficientNetV2` → classificava a imagem inteira (puncionamento, dobra, etc.)
- `YOLOv8` → tentava detectar regiões, mas **não era treinado nos mesmos dados/classes**

**Problema:** O classificador dizia "Puncionamento" com 89%, mas o YOLO não encontrava nada porque foi treinado em outro dataset (NEU-DET, com classes diferentes). Os modelos não conversavam.

## Solução

Treinar **um único YOLOv8** no GC10-DET (com os XMLs Pascal VOC que você já tem), que:
- ✅ Classifica a região detectada (qual defeito é)
- ✅ Localiza com bounding box na imagem
- ✅ Tudo em uma só inferência — sem dois modelos separados

O `app.py` já usa YOLOv8 — basta trocar o `yolo_detector.pt` pelo modelo gerado aqui.

## Estrutura esperada de diretórios
```
/mnt/c/Users/cesar.macieira/Desktop/Trabalho/Python_312/defeitos-superficie/
├── images/images/
│   ├── punching_hole/   *.jpg
│   ├── welding_line/    *.jpg
│   └── ... (10 classes)
└── label/label/         *.xml  (Pascal VOC)
```

## 1. Instalação de dependências

In [1]:
#!pip install ultralytics torch torchvision scikit-learn matplotlib seaborn pillow tqdm pyyaml opencv-python lxml albumentations --quiet

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA GeForce RTX 5090 Laptop GPU
VRAM    : 25.7 GB


## 2. Imports

In [2]:
import os
import glob
import shutil
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from lxml import etree
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports OK")

Imports OK


## 3. Configuração de caminhos

**Não altere os diretórios abaixo** — são os mesmos do seu projeto original.

In [3]:
# ── Caminhos — mantidos idênticos ao gc10det_training.ipynb ──────────────────
DATASET_ROOT = Path(r'/mnt/c/Users/cesar.macieira/Desktop/Trabalho/Python_312/analise-defeitos-superficies')

IMAGE_DIR = DATASET_ROOT / 'images' / 'images'
LABEL_DIR = DATASET_ROOT / 'label' / 'label'

if not LABEL_DIR.exists():
    LABEL_DIR = DATASET_ROOT / 'label'

WORK_DIR = DATASET_ROOT

# Pasta onde será criado o dataset no formato YOLO
# (subpastas images/ e labels/ com splits train/val/test)
YOLO_DATASET_DIR = WORK_DIR / 'yolo_dataset'

# Arquivo de configuração gerado automaticamente
YOLO_YAML = WORK_DIR / 'gc10det.yaml'

# Modelo final exportado para o app Streamlit
OUTPUT_MODEL = WORK_DIR / 'yolo_detector.pt'

print(f'Dataset root : {DATASET_ROOT}  — existe: {DATASET_ROOT.exists()}')
print(f'Images dir   : {IMAGE_DIR}  — existe: {IMAGE_DIR.exists()}')
print(f'Labels dir   : {LABEL_DIR}  — existe: {LABEL_DIR.exists()}')

print('\nSubpastas de imagens (classes):')
for d in sorted(IMAGE_DIR.iterdir()):
    if d.is_dir():
        n = len(list(d.glob('*.jpg')) + list(d.glob('*.bmp')) + list(d.glob('*.png')))
        print(f'  {d.name:<25} → {n} imagem(ns)')

Dataset root : /mnt/c/Users/cesar.macieira/Desktop/Trabalho/Python_312/analise-defeitos-superficies  — existe: True
Images dir   : /mnt/c/Users/cesar.macieira/Desktop/Trabalho/Python_312/analise-defeitos-superficies/images/images  — existe: True
Labels dir   : /mnt/c/Users/cesar.macieira/Desktop/Trabalho/Python_312/analise-defeitos-superficies/label/label  — existe: True

Subpastas de imagens (classes):
  crease                    → 52 imagem(ns)
  crescent_gap              → 226 imagem(ns)
  inclusion                 → 216 imagem(ns)
  oil_spot                  → 204 imagem(ns)
  punching_hole             → 219 imagem(ns)
  rolled_pit                → 31 imagem(ns)
  silk_spot                 → 650 imagem(ns)
  waist folding             → 146 imagem(ns)
  water_spot                → 289 imagem(ns)
  welding_line              → 273 imagem(ns)


## 4. Mapeamento de classes

**Mesmo mapeamento do notebook original** — IDs 0..9, mesma ordem.

In [4]:
CLASS_NAMES = [
    'punching_hole',  # 0
    'welding_line',   # 1
    'crescent_gap',   # 2
    'water_spot',     # 3
    'oil_spot',       # 4
    'silk_spot',      # 5
    'inclusion',      # 6
    'rolled_pit',     # 7
    'crease',         # 8
    'waist_folding',  # 9
]

CLASS_PT = {
    'punching_hole': 'Puncionamento',
    'welding_line':  'Linha de Solda',
    'crescent_gap':  'Fresta Crescente',
    'water_spot':    "Mancha d'Agua",
    'oil_spot':      'Mancha de Oleo',
    'silk_spot':     'Mancha Seda',
    'inclusion':     'Inclusao',
    'rolled_pit':    'Cavidade Laminada',
    'crease':        'Dobra',
    'waist_folding': 'Dobra de Cintura',
}

CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

FOLDER_TO_CLASS = {
    'crescent_gap':  'crescent_gap',  'crescent gap':  'crescent_gap',
    'inclusion':     'inclusion',
    'oil_spot':      'oil_spot',      'oil spot':      'oil_spot',
    'punching_hole': 'punching_hole', 'punching hole': 'punching_hole',
    'rolled_pit':    'rolled_pit',    'rolled pit':    'rolled_pit',
    'silk_spot':     'silk_spot',     'silk spot':     'silk_spot',
    'waist_folding': 'waist_folding', 'waist folding': 'waist_folding',
    'water_spot':    'water_spot',    'water spot':    'water_spot',
    'welding_line':  'welding_line',  'welding line':  'welding_line',
    'crease':        'crease',
}

print(f"Classes ({len(CLASS_NAMES)}):")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {i}: {c} → {CLASS_PT[c]}")

Classes (10):
  0: punching_hole → Puncionamento
  1: welding_line → Linha de Solda
  2: crescent_gap → Fresta Crescente
  3: water_spot → Mancha d'Agua
  4: oil_spot → Mancha de Oleo
  5: silk_spot → Mancha Seda
  6: inclusion → Inclusao
  7: rolled_pit → Cavidade Laminada
  8: crease → Dobra
  9: waist_folding → Dobra de Cintura


## 5. Parser de XMLs Pascal VOC → formato YOLO

O YOLO espera labels no formato:
```
<class_id> <x_center> <y_center> <width> <height>  (tudo normalizado 0..1)
```
Os XMLs do GC10-DET estão em Pascal VOC (`xmin, ymin, xmax, ymax` em pixels), então convertemos aqui.

In [ ]:
def parse_xml_to_yolo(xml_path, class_name):
    """
    Lê XML Pascal VOC e retorna a linha YOLO correspondente.
    Retorna None se o XML for inválido.
    """
    class_id = CLASS_TO_IDX.get(class_name)
    if class_id is None:
        return None

    try:
        xml_text = open(str(xml_path), encoding='utf-8', errors='ignore').read()
        sel = etree.HTML(xml_text)

        width  = int(sel.xpath('//size/width/text()')[0])
        height = int(sel.xpath('//size/height/text()')[0])

        xmin = int(sel.xpath('//bndbox/xmin/text()')[0])
        ymin = int(sel.xpath('//bndbox/ymin/text()')[0])
        xmax = int(sel.xpath('//bndbox/xmax/text()')[0])
        ymax = int(sel.xpath('//bndbox/ymax/text()')[0])

        # Sanidade: bbox deve ser positiva e dentro da imagem
        xmin = max(0, min(xmin, width - 1))
        ymin = max(0, min(ymin, height - 1))
        xmax = max(xmin + 1, min(xmax, width))
        ymax = max(ymin + 1, min(ymax, height))

        # Conversão para formato YOLO (centro + largura/altura normalizados)
        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        bbox_w   = (xmax - xmin) / width
        bbox_h   = (ymax - ymin) / height

        return f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_w:.6f} {bbox_h:.6f}"

    except Exception as e:
        return None


# ── Coleta todos os pares (imagem, label) válidos ─────────────────────────────
label_paths = sorted(glob.glob(str(LABEL_DIR / '*.xml')))
image_paths = sorted(glob.glob(str(IMAGE_DIR / '*' / '*.jpg')))

# Índice de imagens por stem
image_index = {Path(p).stem: p for p in image_paths}

records = []  # lista de dicts {img_path, yolo_line, class_name}
skipped = 0

for lbl_path in tqdm(label_paths, desc='Processando XMLs'):
    file_id = Path(lbl_path).stem
    img_path = image_index.get(file_id)
    if img_path is None:
        skipped += 1
        continue

    class_name_raw = Path(img_path).parent.name
    class_name = FOLDER_TO_CLASS.get(class_name_raw.lower())
    if class_name is None:
        skipped += 1
        continue

    yolo_line = parse_xml_to_yolo(lbl_path, class_name)
    if yolo_line is None:
        skipped += 1
        continue

    records.append({
        'img_path':   img_path,
        'class_name': class_name,
        'yolo_line':  yolo_line,
        'file_id':    file_id,
    })

print(f'Registros válidos : {len(records)}')
print(f'Pulados (sem par) : {skipped}')

# Distribuição de classes
from collections import Counter
dist = Counter(r['class_name'] for r in records)
print('\nDistribuição:')
for cls, n in dist.most_common():
    print(f'  {cls:<25} {n}')

## 6. Exploração visual — amostras com bounding boxes

In [ ]:
# Agrupa por classe
by_class = {}
for r in records:
    by_class.setdefault(r['class_name'], []).append(r)

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for ax, cls in zip(axes, CLASS_NAMES):
    recs = by_class.get(cls, [])
    if not recs:
        ax.axis('off')
        continue
    r = recs[0]
    img = np.array(Image.open(r['img_path']).convert('RGB'))
    h, w = img.shape[:2]

    # Decodifica linha YOLO de volta para pixels para visualização
    parts = list(map(float, r['yolo_line'].split()))
    _, xc, yc, bw, bh = parts
    x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
    x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)

    ax.imshow(img)
    rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f"{CLASS_PT[cls]}\n({len(recs)} imgs)", fontsize=8)
    ax.axis('off')

plt.suptitle('GC10-DET — 1 amostra por classe com bbox (formato YOLO validado)', fontsize=13)
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'amostras_yolo_unificado.png'), dpi=120, bbox_inches='tight')
plt.show()
print("Imagem salva em:", WORK_DIR / 'amostras_yolo_unificado.png')

## 7. Stratified split 70 / 15 / 15

Mesma estratégia do notebook original — garante classes raras em val e test.

In [ ]:
labels_all = [r['class_name'] for r in records]
indices    = list(range(len(records)))

# 1ª divisão: 85% treino+val / 15% teste
idx_tv, idx_test = train_test_split(
    indices, test_size=0.15, stratify=labels_all, random_state=SEED
)

# 2ª divisão: 70% treino / 15% val (do total)
labels_tv = [labels_all[i] for i in idx_tv]
idx_train, idx_val = train_test_split(
    idx_tv, test_size=(0.15/0.85), stratify=labels_tv, random_state=SEED
)

train_recs = [records[i] for i in idx_train]
val_recs   = [records[i] for i in idx_val]
test_recs  = [records[i] for i in idx_test]

print(f'Train : {len(train_recs)} imagens')
print(f'Val   : {len(val_recs)} imagens')
print(f'Test  : {len(test_recs)} imagens')

# Verificação: distribuição por split
print('\nDistribuição por split:')
print(f'  {"Classe":<25} {"Train":>6} {"Val":>6} {"Test":>6}')
tc = Counter(r['class_name'] for r in train_recs)
vc = Counter(r['class_name'] for r in val_recs)
xc = Counter(r['class_name'] for r in test_recs)
for cls in CLASS_NAMES:
    print(f'  {cls:<25} {tc[cls]:>6} {vc[cls]:>6} {xc[cls]:>6}')

## 8. Geração do dataset YOLO no disco

Cria a estrutura de pastas esperada pelo Ultralytics:
```
yolo_dataset/
  images/train/  images/val/  images/test/
  labels/train/  labels/val/  labels/test/
```

In [ ]:
def write_yolo_split(split_records, split_name, yolo_dir):
    img_out = yolo_dir / 'images' / split_name
    lbl_out = yolo_dir / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for r in tqdm(split_records, desc=f'Escrevendo {split_name}'):
        # Copia imagem
        dst_img = img_out / (r['file_id'] + '.jpg')
        if not dst_img.exists():
            shutil.copy2(r['img_path'], dst_img)

        # Escreve label YOLO
        dst_lbl = lbl_out / (r['file_id'] + '.txt')
        dst_lbl.write_text(r['yolo_line'] + '\n', encoding='utf-8')

    print(f'  → {split_name}: {len(split_records)} pares imagem+label')


# Remove dataset anterior se existir
if YOLO_DATASET_DIR.exists():
    shutil.rmtree(YOLO_DATASET_DIR)
    print('Dataset anterior removido.')

write_yolo_split(train_recs, 'train', YOLO_DATASET_DIR)
write_yolo_split(val_recs,   'val',   YOLO_DATASET_DIR)
write_yolo_split(test_recs,  'test',  YOLO_DATASET_DIR)

print(f'\nDataset YOLO em: {YOLO_DATASET_DIR}')

## 9. Geração do arquivo YAML de configuração

In [ ]:
yaml_content = {
    'path':  str(YOLO_DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES,
}

with open(YOLO_YAML, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print(f'YAML salvo em: {YOLO_YAML}')
print()
print(open(YOLO_YAML).read())

## 10. Treinamento YOLOv8 — modelo unificado

### Por que YOLOv8l?
- `yolov8l` (large) tem mAP superior ao `yolov8n/s/m` em datasets pequenos como GC10-DET
- Com GPU, o tempo extra de treino vale o ganho em precisão

### Por que não dois modelos separados?
- Com YOLOv8, **classificação e detecção são a mesma saída**: cada bounding box já vem com a classe e confiança
- O `app.py` já está preparado para isso — o `detect_with_yolo()` lê `box.cls` e `box.conf`

### Hiperparâmetros
- `imgsz=640` — padrão YOLO, funciona bem com GC10-DET
- `epochs=100` — com early stopping ativo (`patience=20`)
- `batch=16` — ajuste conforme sua VRAM (reduza para 8 se tiver <8 GB)
- `augment=True` — augmentação agressiva (mosaic, flipud, degrees, etc.)
- `cos_lr=True` — learning rate cosine annealing
- `label_smoothing=0.1` — ajuda com classes desbalanceadas

> 💡 Se sua GPU tiver menos de 6 GB de VRAM, troque `model='yolov8m.pt'` ou `batch=8`

In [ ]:
from ultralytics import YOLO

# Carrega o backbone pré-treinado no COCO
# yolov8l.pt = Large (melhor mAP, mais lento)
# yolov8m.pt = Medium (bom equilíbrio se VRAM for limitada)
yolo = YOLO('yolov8l.pt')

results = yolo.train(
    data       = str(YOLO_YAML),
    epochs     = 100,
    patience   = 20,          # early stopping se val mAP não melhorar
    imgsz      = 640,
    batch      = 16,          # reduza para 8 se tiver < 8 GB VRAM
    device     = 0,           # GPU 0 (mude para 'cpu' se não tiver GPU)
    workers    = 4,
    project    = str(WORK_DIR / 'runs'),
    name       = 'gc10det_yolo_unificado',
    exist_ok   = True,
    # ── Augmentações ────────────────────────────────────────────────────────
    augment    = True,
    degrees    = 5.0,         # rotação leve (defeitos são sensíveis à orientação)
    fliplr     = 0.5,
    flipud     = 0.0,         # não flipa verticalmente (contexto industrial)
    mosaic     = 1.0,
    mixup      = 0.1,
    hsv_h      = 0.015,
    hsv_s      = 0.5,
    hsv_v      = 0.3,
    # ── Otimização ──────────────────────────────────────────────────────────
    optimizer  = 'AdamW',
    lr0        = 0.001,
    lrf        = 0.01,
    cos_lr     = True,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    label_smoothing = 0.1,    # ajuda com classes desbalanceadas
    # ── Métricas ─────────────────────────────────────────────────────────────
    save_period = 10,
    val         = True,
    plots       = True,
    verbose     = True,
)

print('\n✅ Treinamento concluído!')
print(f'Melhor modelo: {results.save_dir}/weights/best.pt')

## 11. Avaliação no conjunto de teste

In [ ]:
# Caminho do melhor modelo treinado
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'

# Carrega o modelo treinado
model_eval = YOLO(str(best_pt))

# Avalia no conjunto de teste
metrics = model_eval.val(
    data    = str(YOLO_YAML),
    split   = 'test',
    device  = 0,
    imgsz   = 640,
    plots   = True,
    verbose = True,
)

print('\n── Métricas no conjunto de teste ──────────────────────')
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precisão     : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

## 12. Métricas por classe

In [ ]:
# mAP por classe
print(f'  {"Classe":<25} {"mAP@50":>8} {"Precisão":>9} {"Recall":>8}')
print('  ' + '-' * 55)

ap50_per_class = metrics.box.ap50       # array com mAP@50 por classe
p_per_class    = metrics.box.p          # precisão por classe
r_per_class    = metrics.box.r          # recall por classe

for i, cls in enumerate(CLASS_NAMES):
    if i < len(ap50_per_class):
        print(f'  {cls:<25} {ap50_per_class[i]:>8.4f} {p_per_class[i]:>9.4f} {r_per_class[i]:>8.4f}')

# Plot mAP por classe
fig, ax = plt.subplots(figsize=(12, 5))
classes_valid = CLASS_NAMES[:len(ap50_per_class)]
colors = ['#e74c3c' if v < 0.5 else '#3498db' if v < 0.7 else '#2ecc71'
          for v in ap50_per_class]
ax.barh(classes_valid, ap50_per_class, color=colors)
ax.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='mAP=0.5')
ax.set_xlabel('mAP@0.5')
ax.set_title('mAP@0.5 por classe — YOLOv8 Unificado GC10-DET')
ax.legend()
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'map_por_classe.png'), dpi=120)
plt.show()

## 13. Inferência visual — amostras do conjunto de teste

In [ ]:
# Seleciona até 10 imagens aleatórias do conjunto de teste (1 por classe)
test_img_dir = YOLO_DATASET_DIR / 'images' / 'test'
test_images  = list(test_img_dir.glob('*.jpg'))

# 1 imagem por classe para visualização
shown = {}
sample_imgs = []
random.shuffle(test_images)
for img_p in test_images:
    # Descobre a classe pelo label correspondente
    lbl_p = YOLO_DATASET_DIR / 'labels' / 'test' / (img_p.stem + '.txt')
    if lbl_p.exists():
        cls_id = int(lbl_p.read_text().split()[0])
        if cls_id not in shown:
            shown[cls_id] = True
            sample_imgs.append(img_p)
    if len(sample_imgs) >= 10:
        break

# Inferência
preds = model_eval.predict(
    [str(p) for p in sample_imgs],
    conf   = 0.25,
    imgsz  = 640,
    device = 0,
    verbose = False,
)

# Visualização
n = len(preds)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4))
axes = axes.flatten() if rows > 1 else axes

for i, res in enumerate(preds):
    ax = axes[i] if n > 1 else axes
    img_bgr = res.plot(conf=True, labels=True, line_width=2)
    img_rgb = img_bgr[..., ::-1]  # BGR → RGB
    ax.imshow(img_rgb)
    # Título com classes detectadas
    dets = []
    if res.boxes:
        for box in res.boxes:
            cls_id = int(box.cls[0].item())
            conf   = float(box.conf[0].item())
            dets.append(f"{CLASS_PT.get(CLASS_NAMES[cls_id], CLASS_NAMES[cls_id])} {conf*100:.0f}%")
    ax.set_title('\n'.join(dets) if dets else 'Sem detecção', fontsize=8)
    ax.axis('off')

# Esconde eixos sobressalentes
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Inferência YOLOv8 Unificado — Amostras de Teste', fontsize=13)
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'inferencia_amostras.png'), dpi=120, bbox_inches='tight')
plt.show()

## 14. Curvas de treinamento

In [ ]:
# O Ultralytics salva results.csv automaticamente
results_csv = Path(results.save_dir) / 'results.csv'

if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = [c.strip() for c in df_res.columns]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))

    plot_pairs = [
        ('train/box_loss',  'val/box_loss',  'Box Loss',       axes[0, 0]),
        ('train/cls_loss',  'val/cls_loss',  'Class Loss',     axes[0, 1]),
        ('train/dfl_loss',  'val/dfl_loss',  'DFL Loss',       axes[0, 2]),
        ('metrics/mAP50(B)',None,            'mAP@0.50',       axes[1, 0]),
        ('metrics/mAP50-95(B)', None,        'mAP@0.50:0.95',  axes[1, 1]),
        ('metrics/precision(B)', 'metrics/recall(B)', 'Precisão / Recall', axes[1, 2]),
    ]

    for col_a, col_b, title, ax in plot_pairs:
        if col_a in df_res.columns:
            ax.plot(df_res[col_a], label='train' if col_b else col_a.split('/')[-1], color='#3498db')
        if col_b and col_b in df_res.columns:
            ax.plot(df_res[col_b], label='val' if col_a != col_b else col_b.split('/')[-1],
                    color='#e74c3c', linestyle='--')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.suptitle('Curvas de Treinamento — YOLOv8 Unificado GC10-DET', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(WORK_DIR / 'curvas_treinamento.png'), dpi=120)
    plt.show()
else:
    print(f'Arquivo results.csv não encontrado em {results.save_dir}')

## 15. Exportação do modelo para o app Streamlit

Copia o `best.pt` para `yolo_detector.pt` na raiz do projeto — exatamente o caminho que o `app.py` procura.

In [ ]:
import shutil
import json

# ── Copia best.pt → yolo_detector.pt ─────────────────────────────────────────
shutil.copy2(str(best_pt), str(OUTPUT_MODEL))
print(f'✅ Modelo exportado: {OUTPUT_MODEL}')

# ── Atualiza detection_metadata.json ─────────────────────────────────────────
# O app.py lê esse arquivo para o limiar OOD e outras configs
meta_path = WORK_DIR / 'detection_metadata.json'

# Tenta carregar metadata existente para não sobrescrever configs do classificador OOD
if meta_path.exists():
    existing_meta = json.loads(meta_path.read_text(encoding='utf-8'))
else:
    existing_meta = {}

# Adiciona/atualiza campos relacionados ao YOLO unificado
existing_meta.update({
    'yolo_unified': True,
    'yolo_model': 'yolo_detector.pt',
    'yolo_classes': CLASS_NAMES,
    'yolo_map50': float(metrics.box.map50),
    'yolo_map': float(metrics.box.map),
    'ood_threshold': existing_meta.get('ood_threshold', 0.40),  # mantém config OOD se existir
})

meta_path.write_text(json.dumps(existing_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Metadata atualizado: {meta_path}')

print()
print('══════════════════════════════════════════════════════')
print('  MODELO UNIFICADO PRONTO PARA USO NO APP.PY')
print('══════════════════════════════════════════════════════')
print(f'  Arquivo: {OUTPUT_MODEL}')
print(f'  Classes: {len(CLASS_NAMES)} ({", ".join(CLASS_NAMES)})')
print(f'  mAP@0.5: {metrics.box.map50:.4f}')
print()
print('  O app.py já procura "yolo_detector.pt" automaticamente.')
print('  Reinicie o Streamlit e o modelo unificado estará ativo.')
print()
print('  ⚠️  Nota: O app usa APENAS o YOLOv8 agora para detecção.')
print('      O EfficientNetV2 (gc10det_cls_ood.pt) ainda é usado')
print('      para classificação de imagem inteira + OOD detection.')
print('      Ambos os modelos coexistem no pipeline do app.py.')

## 16. (Opcional) Pipeline totalmente unificado: apenas YOLOv8

Se quiser **remover completamente o EfficientNetV2** e usar apenas o YOLOv8 para tudo (classificação + detecção), o trecho abaixo demonstra como fazer a inferência completa com um único modelo.

Para integrar ao `app.py`, você substituiria as funções `classify_image()` e `detect_with_yolo()` por esta função unificada.

In [ ]:
def inferir_unificado(pil_img, conf_threshold=0.25, iou_threshold=0.45):
    """
    Inferência unificada com YOLOv8:
    - Retorna classificação (classe mais confiante) + bounding boxes
    - Substitui os dois modelos separados (EfficientNetV2 + YOLO)
    """
    import tempfile

    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        pil_img.convert('RGB').save(tmp.name, 'JPEG')
        tmp_path = tmp.name

    res_list = model_eval.predict(
        tmp_path,
        conf    = conf_threshold,
        iou     = iou_threshold,
        imgsz   = 640,
        device  = 0,
        verbose = False,
    )
    Path(tmp_path).unlink(missing_ok=True)

    detections = []
    for res in res_list:
        if res.boxes is None:
            continue
        for box in res.boxes:
            cls_id = int(box.cls[0].item())
            score  = float(box.conf[0].item())
            xyxy   = box.xyxy[0].tolist()
            label  = CLASS_NAMES[cls_id]
            detections.append({
                'label':    label,
                'label_pt': CLASS_PT.get(label, label),
                'score':    score,
                'bbox':     xyxy,
                'source':   'YOLOv8-Unificado',
            })

    # Classificação: classe com maior score
    if detections:
        best = max(detections, key=lambda d: d['score'])
        top_label = best['label']
        top_score = best['score']
    else:
        top_label = None
        top_score = 0.0

    return {
        'top_label':   top_label,
        'top_label_pt': CLASS_PT.get(top_label, top_label) if top_label else 'Sem defeito',
        'top_score':   top_score,
        'detections':  detections,
        'n_detections': len(detections),
    }


# ── Teste rápido com uma imagem do conjunto de teste ─────────────────────────
test_sample = sample_imgs[0] if sample_imgs else None

if test_sample:
    img_test = Image.open(test_sample)
    resultado = inferir_unificado(img_test, conf_threshold=0.25)

    print('Resultado da inferência unificada:')
    print(f'  Classe principal : {resultado["top_label_pt"]} ({resultado["top_score"]*100:.1f}%)')
    print(f'  Detecções totais : {resultado["n_detections"]}')
    for d in resultado['detections']:
        print(f'    → {d["label_pt"]:<25} {d["score"]*100:.1f}%  bbox={[round(v) for v in d["bbox"]]}')

    # Visualiza
    from ultralytics import YOLO as _YOLO
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        img_test.save(tmp.name)
        tmp_path = tmp.name
    res = model_eval.predict(tmp_path, conf=0.25, imgsz=640, device=0, verbose=False)[0]
    Path(tmp_path).unlink(missing_ok=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(res.plot()[..., ::-1])
    ax.set_title(f'Inferência unificada: {resultado["top_label_pt"]} ({resultado["top_score"]*100:.1f}%)')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(str(WORK_DIR / 'teste_inferencia_unificada.png'), dpi=120)
    plt.show()
else:
    print('Nenhuma imagem de teste disponível para demo.')

## 17. Resumo final

### O que foi feito
- Lidos os XMLs Pascal VOC do GC10-DET e convertidos para o formato YOLO
- Criado dataset com split estratificado 70/15/15
- Treinado YOLOv8l no GC10-DET: **um único modelo que classifica e localiza**
- Avaliado no conjunto de teste com mAP@0.5
- Exportado como `yolo_detector.pt` no diretório do projeto

### Por que agora vai funcionar no app
Antes:
- `EfficientNetV2` → treinado no GC10-DET ✅
- `YOLOv8` → treinado no **NEU-DET** (dataset diferente, classes diferentes) ❌

Depois deste notebook:
- `EfficientNetV2` → classificação de imagem + OOD ✅ (mantido)
- `YOLOv8` → **treinado no GC10-DET**, mesmas 10 classes ✅ (substituído)

### Próximos passos
1. Reinicie o Streamlit
2. Envie uma imagem com defeito `punching_hole`
3. O YOLOv8 agora deve detectar e marcar as regiões corretamente

### Dicas para melhorar
- **Classes com poucos exemplos** (`rolled_pit`, `crease`): coletar mais imagens ou usar augmentação mais agressiva
- **Aumentar `imgsz=1280`**: pode ajudar defeitos pequenos, mas exige mais VRAM
- **TTA na inferência**: `model.predict(..., augment=True)` para mais robustez